# 06. Comprehensions & Functional Programming Helpers: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **06. Comprehensions & Functional Programming Helpers**. Python provides expressive constructs for collection transformation: list, set, and dictionary comprehensions, alongside generator expressions and functional helpers (`map`, `filter`, `functools.reduce`, `itertools`, `operator`). This notebook covers comprehension filtering and nested iterations, generator memory efficiency, lambda functions, and function composition.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Standard List Comprehension: `[expr for item in iterable]`
- [x] 🔹 Filtered List Comprehension: `[expr for item in iterable if condition]`
- [x] 🔹 Dictionary Comprehensions: `{k_expr: v_expr for item in iterable}`
- [x] 🔹 Set Comprehensions: `{expr for item in iterable}`
- [x] 🔹 Lazy Generator Expressions: `(expr for item in iterable)`
- [x] 🔹 Functional Helper: `map()`
- [x] 🔹 Functional Helper: `filter()`
- [x] 🔹 Functional Helper: `sorted()`
- [x] 🔹 Iteration Tool: `enumerate(iter, start)`
- [x] 🔹 Iteration Tool: `zip(*iters)`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get('transaction_amount'):
            transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Standard List Comprehension: `[expr for item in iterable]`
- **What it does:** Constructs a new list by evaluating an expression for each element in an iterable sequence in a single optimized C-level loop.
- **Syntax:** `[expr for item in iterable]`
- **Key Note:** List comprehensions run significantly faster than equivalent manual `for` loops appending to an empty list due to C-level list allocation optimization.
- **Dataset Application & Code Demonstration:** Applies Standard List Comprehension on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [2]:
amounts = [float(t['transaction_amount'] or 0.0) for t in transactions[:5]]
print('List comprehension amounts:', amounts)

List comprehension amounts: [607.78, 1819.11, 64.08, 1025.73, 772.74]


### 🔹 Filtered List Comprehension: `[expr for item in iterable if condition]`
- **What it does:** Constructs a new list by filtering elements through an inline boolean predicate before evaluating the transformation expression.
- **Syntax:** `[expr for item in iterable if condition]`
- **Key Note:** Placing `if` at the end filters elements. If you need conditional transformations (`expr_if_true if cond else expr_if_false`), place the ternary if-else before `for`.
- **Dataset Application & Code Demonstration:** Applies Filtered List Comprehension on fintech records using columns `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [3]:
high_val_txs = [t['transaction_id'] for t in transactions[:10] if float(t['transaction_amount'] or 0.0) > 500.0]
print('Filtered high-value IDs:', high_val_txs)

Filtered high-value IDs: ['TX109326', 'TX106376', 'TX110701', 'TX103284', 'TX104105', 'TX114584']


### 🔹 Dictionary Comprehensions: `{k_expr: v_expr for item in iterable}`
- **What it does:** Constructs a new dictionary by dynamically generating key-value pairs from an iterable sequence.
- **Syntax:** `{key_expression: value_expression for item in iterable if condition}`
- **Key Note:** If multiple items produce identical keys, later iterations will overwrite earlier values.
- **Dataset Application & Code Demonstration:** Constructs a lookup dictionary mapping `transaction_id` keys to customer and amount details.


In [4]:
tx_map = {t['transaction_id']: float(t['transaction_amount'] or 0.0) for t in transactions[:3]}
print('Dict comprehension map:', tx_map)

Dict comprehension map: {'TX109326': 607.78, 'TX106376': 1819.11, 'TX103301': 64.08}


### 🔹 Set Comprehensions: `{expr for item in iterable}`
- **What it does:** Constructs a new set containing unique, unordered elements evaluated across an iterable.
- **Syntax:** `{expression for item in iterable if condition}`
- **Key Note:** Automatically deduplicates values upon insertion with O(1) average hash table complexity.
- **Dataset Application & Code Demonstration:** Extracts the set of unique merchant category labels present in the transaction dataset.


In [5]:
card_set = {t['card_type'] for t in transactions[:15]}
print('Set comprehension unique cards:', card_set)

Set comprehension unique cards: {'MasterCard', 'Amex', 'Discover', 'Visa'}


### 🔹 Lazy Generator Expressions: `(expr for item in iterable)`
- **What it does:** Creates a lazy generator iterator that computes values on-demand one by one, consuming minimal constant memory `O(1)`.
- **Syntax:** `(expression for item in iterable if condition)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Use generator expressions inside reducers like `sum(...)`, `min(...)`, and `max(...)` to avoid materializing large intermediate lists in RAM.
- **Dataset Application & Code Demonstration:** Computes total revenue using a memory-efficient generator passed directly into `sum()`.


In [6]:
gen_expr = (float(t['transaction_amount'] or 0.0) for t in transactions[:5])
print('Generator object:', gen_expr)
print('Sum of generator items:', round(sum(gen_expr), 2))

Generator object: <generator object <genexpr> at 0x000002343878F140>
Sum of generator items: 4289.44


### 🔹 Functional Helper: `map()`
- **What it does:** Applies a transformation function to each element of an iterable lazily.
- **Syntax:** `map()`
- **Key Note:** `.apply()` runs a standard Python loop row-by-row. Whenever possible, use built-in vectorized operations (`df['a'] + df['b']`) which run up to 100x faster!.
- **Dataset Application & Code Demonstration:** Applies Functional Helper on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [7]:
mapped_amounts = list(map(lambda t: float(t['transaction_amount'] or 0.0), transactions[:4]))
print('map() converted amounts:', mapped_amounts)

map() converted amounts: [607.78, 1819.11, 64.08, 1025.73]


### 🔹 Functional Helper: `filter()`
- **What it does:** Filters elements of an iterable yielding only items where predicate returns True.
- **Syntax:** `filter()`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Functional Helper on fintech records using columns `is_fraud` to demonstrate real-world execution.


In [8]:
fraud_records = list(filter(lambda t: int(t['is_fraud']) == 1, transactions[:20]))
print(f'filter() extracted {len(fraud_records)} fraud records.')

filter() extracted 3 fraud records.


### 🔹 Functional Helper: `sorted()`
- **What it does:** Returns a new sorted list using custom key functions and reverse flag.
- **Syntax:** `sorted()`
  - **Optional Parameters:**
    - `key` (*callable, optional*): Function of one argument to extract comparison key.
    - `reverse` (*bool, default False*): If True, sort elements in descending order.
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Functional Helper on fintech records using columns `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [9]:
sorted_top3 = sorted(transactions[:5], key=lambda t: float(t['transaction_amount'] or 0.0), reverse=True)[:3]
print('Top 3 sorted transactions:')
for t in sorted_top3:
    print(f"{t['transaction_id']}: ${float(t['transaction_amount'] or 0.0):.2f}")

Top 3 sorted transactions:
TX106376: $1819.11
TX110701: $1025.73
TX103284: $772.74


### 🔹 Iteration Tool: `enumerate(iter, start)`
- **What it does:** Yields tuples containing running count index and item.
- **Syntax:** `enumerate(iter, start)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.
- **Dataset Application & Code Demonstration:** Applies Iteration Tool on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [10]:
for rank, t in enumerate(transactions[:3], start=1):
    print(f'Rank #{rank}: {t["transaction_id"]}')

Rank #1: TX109326
Rank #2: TX106376
Rank #3: TX103301


### 🔹 Iteration Tool: `zip(*iters)`
- **What it does:** Aggregates corresponding elements from multiple iterables into tuples.
- **Syntax:** `zip(*iters)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.
- **Dataset Application & Code Demonstration:** Applies Iteration Tool on fintech records using columns `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [11]:
ids = [t['transaction_id'] for t in transactions[:3]]
amts = [float(t['transaction_amount'] or 0.0) for t in transactions[:3]]
for tx_id, amt in zip(ids, amts):
    print(f'Zipped pair: {tx_id} -> ${amt:.2f}')

Zipped pair: TX109326 -> $607.78
Zipped pair: TX106376 -> $1819.11
Zipped pair: TX103301 -> $64.08


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Generator Expression vs List Comprehension Memory
- **Objective:** Q1: Generator Expression vs List Comprehension Memory
- **Approach:** Demonstrate memory scaling difference on 15,000 transactions.
- **Syntax:** `sys.getsizeof([x for x in data])` vs `sys.getsizeof(x for x in data)`

In [12]:
lst_comp = [float(t['transaction_amount'] or 0.0) for t in transactions]
gen_expr = (float(t['transaction_amount'] or 0.0) for t in transactions)
print(f'List Comprehension RAM: {sys.getsizeof(lst_comp) / 1024:.1f} KB')
print(f'Generator Expression RAM: {sys.getsizeof(gen_expr)} bytes (constant O(1))')

List Comprehension RAM: 118.6 KB
Generator Expression RAM: 216 bytes (constant O(1))
